# Step 9-A — Per-query re-rank + second generation

**Purpose:** produce the two missing artifacts identified in `step9_plan.md` §2/§12:
a per-query re-ranked Top-10 list (for both institution-aware MMR and Fair-Top-K),
and a second Gemini generation on that re-ranked context, for the Tier 1
(~20-query) scope defined in `step9_query_subset.md` §2.

**Cost note (corrected 2026-08-03):** the original plan estimated ~20 extra
Gemini calls for Tier 1, based on a single re-ranking method. Since both
MMR and Fair-Top-K now exist and get demoed, and only neutral queries carry
an intervention panel (§10b), the actual count is **14 neutral queries x
2 methods = 28 calls**. Contradictory queries (6 in scope) get zero
intervention calls by design.

**Kaggle datasets to add to this notebook** (Add Data -> search by owner
`yanbochen928`), matching what Step 6/7a already used:

| Dataset (owner: yanbochen928) | Files used here |
|---|---|
| `fairsearch-qbio-embeddings-raj-jici-yb` | `qbio_embeddings.npy`, `embedding_info.json` |
| `fairsearch-qbio-processed-raj-jici-yb` | `qbio_papers.json` |
| `fairsearch-qbio-1b-labels-raj-jici-yb` | `sample_labels_1000.json`, `retrieval_labels.json`, `candidate_labels.json` |
| `fairsearch-qbio-queries-yb` | `retrieval_results.json` |
| `fairsearch-qbio-elite-list` | `qs_top50_elite_2026.json` |

Also add the **queries dataset that has `queries_all_150.json`** (same one
Step 5/7 use), and if you have `data/candidate_labels.json` published as a
Kaggle Dataset from the Step 6 session, add that too -- it already carries
Option-B labels for every candidate-pool paper the Step 6 pipeline touched,
which should cover this subset's candidate pools too (same query
population), so **no OpenAlex re-querying should be needed in this
notebook**. If `find()` below can't locate it, upload
`data/candidate_labels.json` as a new one-off Kaggle Dataset first.

**Internet must be ON** (SDK install + Gemini calls) and the
`GEMINI_API_KEY` secret must be attached, same as Step 7a/7b.

Work through this one cell at a time. Paste any error back before
continuing to the next cell.

## Cell 0 - Environment check: Gemini SDK

Verbatim from Step 7a Cell 0. Checks for `google-genai`, installs it if
missing. No API call yet.

In [1]:
# Cell 0: check + install Gemini SDK (no API call yet)
import importlib, subprocess, sys

def pkg_version(mod_name):
    try:
        m = importlib.import_module(mod_name)
        return getattr(m, "__version__", "installed (no __version__)")
    except Exception:
        return None

old_sdk = pkg_version("google.generativeai")
new_sdk = pkg_version("google.genai")
print("BEFORE install:")
print("  google-generativeai (old):", old_sdk)
print("  google-genai (new)       :", new_sdk)

if new_sdk is None:
    print("\ngoogle-genai not found -> installing ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai"], check=True)
    importlib.invalidate_caches()
    new_sdk = pkg_version("google.genai")
else:
    print("\ngoogle-genai already present -> no install needed")

print("\nAFTER install:")
print("  google-genai (new):", new_sdk)
assert new_sdk is not None, "google-genai still not importable; check Internet=ON"
print("\nOK: google-genai is available. Proceed to Cell 1.")


/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)


BEFORE install:
  google-generativeai (old): 0.8.6
  google-genai (new)       : 1.68.0

google-genai already present -> no install needed

AFTER install:
  google-genai (new): 1.68.0

OK: google-genai is available. Proceed to Cell 1.


## Cell 1 - Read API key, init client, smoke test

Verbatim pattern from Step 7a Cell 2. Same model and temperature as Step
7a/7b, so the only variable changing between baseline and intervention is
the retrieved context, not the model or its settings (per `step9_plan.md`
§11, 9-0).

In [2]:
# Cell 1: read key, init client, smoke test (one tiny call)
from kaggle_secrets import UserSecretsClient
from google import genai
from google.genai import types

GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
GEMINI_API_KEY = GEMINI_API_KEY.strip() if GEMINI_API_KEY else GEMINI_API_KEY
assert GEMINI_API_KEY, "key not found; check Secrets label = GEMINI_API_KEY and that it is attached"
print("key loaded from Secrets: OK (length =", len(GEMINI_API_KEY), ")")

client = genai.Client(api_key=GEMINI_API_KEY)

GEN_MODEL = "gemini-3.1-flash-lite"     # SAME as Step 7a/7b
GEN_TEMPERATURE = 0                     # SAME as Step 7a/7b

resp = client.models.generate_content(
    model=GEN_MODEL,
    contents="Reply with exactly the word: OK",
    config=types.GenerateContentConfig(temperature=GEN_TEMPERATURE),
)
print("model:", GEN_MODEL, "| temperature:", GEN_TEMPERATURE)
print("Gemini replied:", repr(resp.text))
print("\nSmoke test done. If you see a reply above, proceed to Cell 2.")


key loaded from Secrets: OK (length = 53 )
model: gemini-3.1-flash-lite | temperature: 0
Gemini replied: 'OK'

Smoke test done. If you see a reply above, proceed to Cell 2.


## Cell 2 - Config and `find()`

Same config constants and the same Kaggle/local file-locator pattern as
`step6-reranking-yb-optimized-basedon-jici.ipynb` §0, so this notebook
resolves inputs the same way whether run on Kaggle or with local files.

In [3]:
# Cell 2: config + find()
import json, os, glob, re, time
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

rng = np.random.default_rng(42)

BASELINE_SHARE = 0.144
N_CANDIDATES   = 50
K              = 10
EMB_MODEL      = 'all-MiniLM-L6-v2'

KAGGLE_SLUGS = {
    'qbio_embeddings.npy':      'yanbochen928/fairsearch-qbio-embeddings-raj-jici-yb',
    'embedding_info.json':      'yanbochen928/fairsearch-qbio-embeddings-raj-jici-yb',
    'qbio_papers.json':         'yanbochen928/fairsearch-qbio-processed-raj-jici-yb',
    'sample_labels_1000.json':  'yanbochen928/fairsearch-qbio-1b-labels-raj-jici-yb',
    'retrieval_labels.json':    'yanbochen928/fairsearch-qbio-1b-labels-raj-jici-yb',
    'retrieval_results.json':   'yanbochen928/fairsearch-qbio-queries-yb',
    'queries_all_150.json':     'yanbochen928/fairsearch-qbio-queries-yb',
    'qs_top50_elite_2026.json': 'yanbochen928/fairsearch-qbio-elite-list',
    'candidate_labels.json':    'yanbochen928/fairsearch-qbio-1b-labels-raj-jici-yb',
}
_kh_roots = []

def find(hint):
    for base in ['/kaggle/input', 'data', '.', '..', *_kh_roots]:
        hits = glob.glob(f'{base}/**/{hint}', recursive=True)
        if hits:
            return hits[0]
    slug = KAGGLE_SLUGS.get(hint)
    if slug:
        import kagglehub
        root = kagglehub.dataset_download(slug)
        _kh_roots.append(root)
        hits = glob.glob(f'{root}/**/{hint}', recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(hint)

def norm(s): return s.strip().lower() if s else ''

print('Config ready. BASELINE_SHARE =', BASELINE_SHARE, '| K =', K, '| N_CANDIDATES =', N_CANDIDATES)

Config ready. BASELINE_SHARE = 0.144 | K = 10 | N_CANDIDATES = 50


## Cell 3 - Load embeddings, corpus, queries, elite list

Same loading pattern as Step 6 §1. Loads the full 150-query set here
(not just neutral), since Cell 5's sampling step needs both neutral and
contradictory pools to draw from.

In [4]:
# Cell 3: load embeddings + corpus + queries + elite list
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(EMB_MODEL)

emb = np.load(find('qbio_embeddings.npy'))
with open(find('embedding_info.json')) as f:
    einfo = json.load(f)
if isinstance(einfo, list):
    emb_ids = [str(x) for x in einfo]
elif isinstance(einfo, dict):
    emb_ids = [str(x) for x in (einfo.get('paper_ids') or einfo.get('ids') or einfo.get('order'))]
else:
    raise ValueError('unexpected embedding_info.json shape')
assert len(emb_ids) == emb.shape[0], (len(emb_ids), emb.shape)
row_of = {pid: i for i, pid in enumerate(emb_ids)}
embn = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12)
print(f'embeddings: {emb.shape} | ids: {len(emb_ids)}')

with open(find('qbio_papers.json')) as f:
    corpus = json.load(f)
cats_of  = {str(p['paper_id']): str(p.get('categories','')) for p in corpus}
title_of = {str(p['paper_id']): str(p.get('title','')) for p in corpus}

with open(find('retrieval_results.json')) as f:
    retrieval_results = json.load(f)
retrieval_by_id = {q['query_id']: q for q in retrieval_results}
print('retrieval_results.json queries:', len(retrieval_results))

with open(find('queries_all_150.json')) as f:
    queries_all = json.load(f)
neutral_all = [q for q in queries_all if q['type'] == 'neutral']
contra_all  = [q for q in queries_all if q['type'] != 'neutral']
print('queries_all_150.json: neutral', len(neutral_all), '| contradictory', len(contra_all))

with open(find('qs_top50_elite_2026.json')) as f:
    elite_list = json.load(f)
elite_names = {norm(e['name']) for e in elite_list}
print('elite names:', len(elite_names))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embeddings: (55301, 384) | ids: 55301
retrieval_results.json queries: 150
queries_all_150.json: neutral 100 | contradictory 50
elite names: 50


## Cell 4 - Load Option B labels (`label_of`)

Merges the same three label files Step 6 used, INCLUDING
`data/candidate_labels.json` (the file Step 6 produced with real OpenAlex
lookups for every candidate-pool paper across all 100 neutral queries).
Since this notebook's neutral scope (Cell 5) is a subset of those same 100
queries, this merge should already cover every candidate this notebook
touches -- **no new OpenAlex calls should be necessary.** If Cell 6/7 later
reports missing labels, stop and check whether `candidate_labels.json` was
actually loaded (see the printed count below) before writing any new
OpenAlex code.

In [5]:
# Cell 4: build label_of from all three existing label files
label_of = {}
for fn in ['sample_labels_1000.json', 'retrieval_labels.json']:
    with open(find(fn)) as f:
        for r in json.load(f)['labels']:
            label_of[str(r['paper_id'])] = {
                'coverage': r['coverage'],
                'institution': r.get('institution'),
                'country': r.get('country'),
                'elite_label': r.get('elite_label'),
            }
n_after_1b = len(label_of)
print(f'after sample_labels_1000 + retrieval_labels: {n_after_1b:,}')

try:
    with open(find('candidate_labels.json')) as f:
        cand = json.load(f)
    for r in cand['labels']:
        label_of[str(r['paper_id'])] = {
            'coverage': r['coverage'],
            'institution': r.get('institution'),
            'country': r.get('country'),
            'elite_label': r.get('elite_label'),
        }
    print(f'after candidate_labels.json:            {len(label_of):,} (+{len(label_of)-n_after_1b:,})')
except FileNotFoundError:
    print('WARNING: candidate_labels.json not found. Some candidates in this '
          'subset may end up unlabeled that Step 6 already resolved. '
          'Upload data/candidate_labels.json as a Kaggle Dataset input before continuing.')


after sample_labels_1000 + retrieval_labels: 2,345
after candidate_labels.json:            5,431 (+3,086)


## Cell 5 - Sample the Tier 1 ~20-query scope

Implements `step9_query_subset.md` §2 exactly: 14 neutral (stratified,
>=1 per subcategory, q033 retained as a disclosed anchor regardless of the
draw) + 6 contradictory (simple seeded random draw). Seed=42, matching the
seed used throughout RQ1/RQ2/RQ3's bootstrap resampling.

**Disclosed simplification vs. the plan doc:** `step9_query_subset.md`
§2.3 calls for the contradictory slice to be spread across *distinct
debate topics*, but notes this "is a qualitative judgment, not an
algorithmic stratification... debate-topic groupings are not currently
encoded as a field." This cell does a plain seeded random draw of 6 from
50 instead of topic-aware stratification, since no topic field exists to
stratify on. If you want topic spread guaranteed, this is the place to add
a manual topic-group dict before sampling -- flagging it here rather than
silently deciding either way.

Verified locally before writing this cell (see chat) with the exact same
seed and logic: produces 14 neutral queries covering all 10 subcategories
plus q033.

In [6]:
# Cell 5: sample the ~20-query Tier 1 scope
import random

TARGET_NEUTRAL = 14
TARGET_CONTRA  = 6
ANCHOR = 'q033'
SEED = 42
rng_sample = random.Random(SEED)

by_subcat = defaultdict(list)
for q in neutral_all:
    by_subcat[q['subcategory']].append(q['query_id'])

anchor_subcat = next(q['subcategory'] for q in neutral_all if q['query_id'] == ANCHOR)

selected = {ANCHOR}
for sc in sorted(by_subcat):
    if sc == anchor_subcat:
        continue
    pool = [qid for qid in by_subcat[sc] if qid not in selected]
    selected.add(rng_sample.choice(pool))

neutral_ids_all = [q['query_id'] for q in neutral_all]
remaining = TARGET_NEUTRAL - len(selected)
pool_all = [qid for qid in neutral_ids_all if qid not in selected]
selected.update(rng_sample.sample(pool_all, remaining))

QUERY_SCOPE_NEUTRAL = sorted(selected)

contra_ids_all = [q['query_id'] for q in contra_all]
QUERY_SCOPE_CONTRA = sorted(rng_sample.sample(contra_ids_all, TARGET_CONTRA))

QUERY_SCOPE = QUERY_SCOPE_NEUTRAL + QUERY_SCOPE_CONTRA
print(f'neutral scope ({len(QUERY_SCOPE_NEUTRAL)}):', QUERY_SCOPE_NEUTRAL)
print(f'contradictory scope ({len(QUERY_SCOPE_CONTRA)}):', QUERY_SCOPE_CONTRA)
print('anchor q033 included:', ANCHOR in QUERY_SCOPE_NEUTRAL)

subcats_covered = {q['subcategory'] for q in neutral_all if q['query_id'] in QUERY_SCOPE_NEUTRAL}
print('subcategories covered:', len(subcats_covered), '/', len(by_subcat))

scope_payload = {
    'meta': {'seed': SEED, 'target_neutral': TARGET_NEUTRAL, 'target_contra': TARGET_CONTRA,
              'anchor': ANCHOR, 'contra_sampling': 'plain seeded random, NOT topic-stratified (see cell markdown)'},
    'neutral': QUERY_SCOPE_NEUTRAL,
    'contradictory': QUERY_SCOPE_CONTRA,
}
os.makedirs('results', exist_ok=True)
with open('step9_query_scope.json', 'w') as f:
    json.dump(scope_payload, f, indent=2)
print('saved step9_query_scope.json')


neutral scope (14): ['q005', 'q010', 'q013', 'q018', 'q022', 'q033', 'q038', 'q043', 'q050', 'q055', 'q063', 'q071', 'q085', 'q086']
contradictory scope (6): ['q102', 'q106', 'q114', 'q115', 'q133', 'q139']
anchor q033 included: True
subcategories covered: 10 / 10
saved step9_query_scope.json


## Cell 6 - Build candidate pools for the neutral queries in scope

Same seeding pattern as Step 6 §3: each pool is seeded with the stored
Step 5a Top-10 (so lambda=1.0 / the un-reranked baseline reproduces it
exactly), then filled to `N_CANDIDATES` with the nearest embedding
neighbours. Restricted to `QUERY_SCOPE_NEUTRAL` only -- contradictory
queries never get an intervention panel (§10b), so they need no
candidate pool here.

In [7]:
# Cell 6: build candidate pools, neutral scope only
neutral_scope_records = [q for q in neutral_all if q['query_id'] in QUERY_SCOPE_NEUTRAL]
neutral_scope_records.sort(key=lambda q: QUERY_SCOPE_NEUTRAL.index(q['query_id']))

q_emb = model.encode([q['query_text'] for q in neutral_scope_records], normalize_embeddings=True)

pools = []
for qi, q in enumerate(neutral_scope_records):
    sims = embn @ q_emb[qi]
    top = np.argpartition(-sims, N_CANDIDATES)[:N_CANDIDATES]
    top = top[np.argsort(-sims[top])]
    emb_order = [emb_ids[r] for r in top]
    seed = [pid for pid in retrieval_by_id[q['query_id']]['retrieved_paper_ids'] if pid in row_of]
    seen = set(seed)
    ordered = (seed + [pid for pid in emb_order if pid not in seen])[:N_CANDIDATES]
    n = len(ordered)
    pools.append([(pid, 1.0 - idx / n, row_of[pid]) for idx, pid in enumerate(ordered)])

print('built', len(pools), 'candidate pools of size', N_CANDIDATES)

need = sorted({pid for pool in pools for (pid, _, _) in pool if pid not in label_of})
print('candidates still unlabeled after Cell 4 merge:', len(need))
if need:
    print('WARNING: the candidate_labels.json merge in Cell 4 did not cover everything.')
    print('First few unlabeled pids:', need[:10])
    print('Decide before continuing: re-query OpenAlex for just these (small), or accept as unknown.')


built 14 candidate pools of size 50
candidates still unlabeled after Cell 4 merge: 0


## Cell 7 - Re-ranking functions (institution-aware MMR + Fair-Top-K)

Verbatim from `step6-reranking-yb-optimized-basedon-jici.ipynb` §7b and
§7e. `ELITE_QUOTA` and `BASELINE_SHARE` are unchanged from Step 6, so the
Fair-Top-K quota rule here is identical to the one already measured and
written up in `rq3_methodology.md` §8a.

In [8]:
# Cell 7: re-ranking functions, verbatim from Step 6
INST_LAMBDA = 0.8   # pre-registered operating point, rq3_methodology.md §4.2
ELITE_QUOTA = round(K * BASELINE_SHARE)   # = 1

def group(pid):
    lab = label_of.get(pid)
    if lab and lab['coverage'] == 'found':
        return 'elite' if lab['elite_label'] == 1 else 'non_elite'
    return 'unknown'

def rerank_labelaware(pool, lam, field='institution', graded=True):
    cand = [(pid, rel) for (pid, rel, row) in pool]
    def grp(pid):
        lab = label_of.get(pid)
        if lab and lab['coverage'] == 'found' and lab.get(field):
            return lab[field]
        return None
    selected, seen = [], Counter()
    while cand and len(selected) < K:
        best_i, best_score = None, -1e9
        for i, (pid, rel) in enumerate(cand):
            g = grp(pid)
            if g is None:
                penalty = 0.0
            else:
                penalty = float(seen[g]) if graded else (1.0 if seen[g] > 0 else 0.0)
            s = lam * rel - (1 - lam) * penalty
            if s > best_score:
                best_score, best_i = s, i
        pid, rel = cand.pop(best_i)
        selected.append(pid)
        g = grp(pid)
        if g is not None:
            seen[g] += 1
    return selected

def rerank_fair_top_k(pool):
    elite_q     = [pid for (pid, rel, row) in pool if group(pid) == 'elite']
    non_elite_q = [pid for (pid, rel, row) in pool if group(pid) == 'non_elite']
    unknown_q   = [pid for (pid, rel, row) in pool if group(pid) == 'unknown']

    selected = []
    n_elite_seats = min(ELITE_QUOTA, len(elite_q))
    selected.extend(elite_q[:n_elite_seats])
    elite_leftover = elite_q[n_elite_seats:]

    remaining = K - len(selected)
    selected.extend(non_elite_q[:remaining])

    remaining = K - len(selected)
    if remaining > 0:
        selected.extend(elite_leftover[:remaining])
        remaining = K - len(selected)

    if remaining > 0:
        selected.extend(unknown_q[:remaining])
    return selected[:K]

print('Re-ranking functions loaded. ELITE_QUOTA =', ELITE_QUOTA, '| INST_LAMBDA =', INST_LAMBDA)


Re-ranking functions loaded. ELITE_QUOTA = 1 | INST_LAMBDA = 0.8


## Cell 8 - Apply both methods to every neutral-scope query

Produces the per-query artifact `step9_plan.md` §2/§12 identified as
missing: the actual re-ranked Top-10 paper list per query, per method, with
full metadata (title, institution, coverage, elite_label). No API calls in
this cell.

In [9]:
# Cell 8: run both re-rankers per query, assemble per-query records
def paper_record(rank, pid):
    lab = label_of.get(pid, {})
    return {
        'rank': rank,
        'paper_id': pid,
        'title': title_of.get(pid, ''),
        'institution': lab.get('institution'),
        'country': lab.get('country'),
        'coverage': lab.get('coverage', 'not_found'),
        'elite_label': lab.get('elite_label'),
    }

rerank_per_query = {}
for q, pool in zip(neutral_scope_records, pools):
    baseline_pids = [pid for (pid, _, _) in pool[:K]]
    mmr_pids      = rerank_labelaware(pool, INST_LAMBDA, field='institution', graded=True)
    ftk_pids      = rerank_fair_top_k(pool)
    rerank_per_query[q['query_id']] = {
        'query_text': q['query_text'],
        'subcategory': q['subcategory'],
        'baseline': [paper_record(i+1, pid) for i, pid in enumerate(baseline_pids)],
        'institution_aware_mmr': [paper_record(i+1, pid) for i, pid in enumerate(mmr_pids)],
        'fair_top_k': [paper_record(i+1, pid) for i, pid in enumerate(ftk_pids)],
    }

print('assembled per-query re-rank records for', len(rerank_per_query), 'queries')
sample_qid = QUERY_SCOPE_NEUTRAL[0]
print(f'\nsample ({sample_qid}) baseline top-3:')
for p in rerank_per_query[sample_qid]['baseline'][:3]:
    print(' ', p['rank'], p['paper_id'], '|', p['coverage'], '| elite=', p['elite_label'])

with open('step9_rerank_per_query.json', 'w') as f:
    json.dump(rerank_per_query, f, indent=2)
print('\nsaved step9_rerank_per_query.json')


assembled per-query re-rank records for 14 queries

sample (q005) baseline top-3:
  1 1904.11372 | found | elite= 0
  2 1612.06267 | no_affiliation | elite= None
  3 2006.00688 | found | elite= 0

saved step9_rerank_per_query.json


## Cell 9 - Generation prompt + context/citation functions

Verbatim from Step 7a Cells 3-5 (`build_context`, `PROMPT_TEMPLATE`,
`generate_answer`, `parse_citations`), generalised so `build_context` takes
an explicit paper-id list instead of reading `retrieved_paper_ids` off the
query record -- this is the only change, needed because we now call it
once per method per query, not once per query. The prompt text itself is
byte-identical to Step 7a/7b.

In [10]:
# Cell 9: generation + citation-parsing functions (adapted from Step 7a)
def build_context(paper_ids):
    lines = []
    num2pid = {}
    for i, pid in enumerate(paper_ids, start=1):
        num2pid[i] = pid
        title = title_of.get(pid) or '(title unavailable)'
        abstract = next((p.get('abstract','') for p in corpus if str(p['paper_id']) == pid), '') or '(abstract unavailable)'
        lines.append(f"[{i}] Title: {title}\nAbstract: {abstract}")
    return "\n\n".join(lines), num2pid

PROMPT_TEMPLATE = """You are answering a scientific question using ONLY the papers provided below.

Rules:
- Use ONLY information from the provided papers. Do not add outside knowledge.
- After each scientific claim, cite the supporting source number(s) in square
  brackets, e.g. \"... increases noise [3].\" or \"... as shown in [2][5].\"
- For multiple sources, use adjacent markers like [2][5]. Do NOT use [2, 5]
  or ranges like [2-5].
- Every claim must have at least one citation. Cite only from [1]..[10].
- Write a concise answer (roughly one paragraph).

Question:
{question}

Papers:
{context}

Answer (with [n] citations):"""

def generate_answer(query_text, paper_ids):
    context_text, num2pid = build_context(paper_ids)
    prompt = PROMPT_TEMPLATE.format(question=query_text, context=context_text)
    resp = client.models.generate_content(
        model=GEN_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=GEN_TEMPERATURE),
    )
    return resp.text, num2pid

def parse_citations(answer_text, num2pid):
    groups = re.findall(r"\[([0-9,\s]+)\]", answer_text)
    cited_numbers_raw = []
    for g in groups:
        for tok in g.split(','):
            tok = tok.strip()
            if tok.isdigit():
                cited_numbers_raw.append(int(tok))
    valid_numbers, invalid_markers = [], []
    for n in cited_numbers_raw:
        (valid_numbers if n in num2pid else invalid_markers).append(n)
    cited_pids_raw = [num2pid[n] for n in valid_numbers]
    cited_pids_unique = sorted(set(cited_pids_raw))
    return {
        'cited_numbers_raw': cited_numbers_raw,
        'valid_numbers': valid_numbers,
        'invalid_markers': invalid_markers,
        'cited_pids_raw': cited_pids_raw,
        'cited_pids_unique': cited_pids_unique,
    }

print('Generation functions ready. Model:', GEN_MODEL, '| temperature:', GEN_TEMPERATURE)


Generation functions ready. Model: gemini-3.1-flash-lite | temperature: 0


## ⚠️ API WARNING before Cell 10

**This cell spends quota: 14 neutral queries x 2 methods = 28 Gemini
calls.** Nothing before this point calls the API except the Cell 1 smoke
test. Checkpointed per (query, method) pair to
`step9_gen_checkpoint.jsonl`, same pattern as `rq2_gen_checkpoint.jsonl`, so
a partial run can resume without repeating completed calls or losing
progress on a 429.

In [11]:
# Cell 10: second generation, checkpointed (SPENDS QUOTA: ~28 calls)
CKPT_PATH = 'step9_gen_checkpoint.jsonl'
done = set()
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        for line in f:
            rec = json.loads(line)
            done.add((rec['query_id'], rec['method']))
    print(f'resuming: {len(done)} (query, method) pairs already done')

methods = ['institution_aware_mmr', 'fair_top_k']
todo = [(qid, m) for qid in QUERY_SCOPE_NEUTRAL for m in methods if (qid, m) not in done]
print(f'{len(todo)} calls remaining out of {len(QUERY_SCOPE_NEUTRAL) * len(methods)} total')

with open(CKPT_PATH, 'a') as f:
    for qid, method in todo:
        rec = rerank_per_query[qid]
        paper_ids = [p['paper_id'] for p in rec[method]]
        try:
            answer_text, num2pid = generate_answer(rec['query_text'], paper_ids)
        except Exception as e:
            print(f'  [FAIL] {qid} / {method}: {e}')
            continue
        parsed = parse_citations(answer_text, num2pid)
        out = {
            'query_id': qid, 'method': method,
            'answer_text': answer_text, 'num2pid': num2pid,
            'cited_pids_unique': parsed['cited_pids_unique'],
            'invalid_markers': parsed['invalid_markers'],
        }
        f.write(json.dumps(out) + '\n')
        f.flush()
        print(f'  done: {qid} / {method}')
        time.sleep(0.3)

print('\nCell 10 complete.')


28 calls remaining out of 28 total
  done: q005 / institution_aware_mmr
  done: q005 / fair_top_k
  done: q010 / institution_aware_mmr
  done: q010 / fair_top_k
  done: q013 / institution_aware_mmr
  done: q013 / fair_top_k
  done: q018 / institution_aware_mmr
  done: q018 / fair_top_k
  done: q022 / institution_aware_mmr
  done: q022 / fair_top_k
  done: q033 / institution_aware_mmr
  done: q033 / fair_top_k
  done: q038 / institution_aware_mmr
  done: q038 / fair_top_k
  done: q043 / institution_aware_mmr
  done: q043 / fair_top_k
  done: q050 / institution_aware_mmr
  done: q050 / fair_top_k
  done: q055 / institution_aware_mmr
  done: q055 / fair_top_k
  done: q063 / institution_aware_mmr
  done: q063 / fair_top_k
  done: q071 / institution_aware_mmr
  done: q071 / fair_top_k
  done: q085 / institution_aware_mmr
  done: q085 / fair_top_k
  done: q086 / institution_aware_mmr
  done: q086 / fair_top_k

Cell 10 complete.


## Cell 11 - Summary and next steps

Confirms what this notebook produced and what still needs to happen in
9-B (bundle assembly, still to be written) before Streamlit (9-C) can read
real data.

In [12]:
# Cell 11: summary
print('step9_query_scope.json       :', os.path.exists('step9_query_scope.json'))
print('step9_rerank_per_query.json  :', os.path.exists('step9_rerank_per_query.json'))
print('step9_gen_checkpoint.jsonl   :', os.path.exists('step9_gen_checkpoint.jsonl'))

n_lines = 0
if os.path.exists('step9_gen_checkpoint.jsonl'):
    with open('step9_gen_checkpoint.jsonl') as f:
        n_lines = sum(1 for _ in f)
print(f'generation checkpoint records: {n_lines} (expected {len(QUERY_SCOPE_NEUTRAL) * 2})')

print('\nNext: Step 9-B assembles step9_query_scope.json + step9_rerank_per_query.json')
print('+ step9_gen_checkpoint.jsonl + baseline generation (rq2_gen_checkpoint.jsonl,')
print('rq2_frameworkB_generation_raw.jsonl) + RQ1/RQ2 diagnostics into step9_bundle.json,')
print('per step9_plan.md §10a schema.')


step9_query_scope.json       : True
step9_rerank_per_query.json  : True
step9_gen_checkpoint.jsonl   : True
generation checkpoint records: 28 (expected 28)

Next: Step 9-B assembles step9_query_scope.json + step9_rerank_per_query.json
+ step9_gen_checkpoint.jsonl + baseline generation (rq2_gen_checkpoint.jsonl,
rq2_frameworkB_generation_raw.jsonl) + RQ1/RQ2 diagnostics into step9_bundle.json,
per step9_plan.md §10a schema.
